# What factors are associated with Airbnb price?
Exploratory analysis of the **New York City Airbnb Open Data** dataset (Inside Airbnb detailed listings).

Dataset: about 30k active NYC listings, one row per listing, from an Inside Airbnb snapshot.

**Goal:** clean the data and use visualizations to see what is associated with the nightly `price`.


In [ ]:
# --- Imports ---
# pandas/numpy for data handling, matplotlib/seaborn for plotting.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# A clean, consistent look for every chart so they read as one set.
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110          # sharper inline figures
plt.rcParams["axes.titleweight"] = "bold" # bold titles for readability

# One row per listing. This is the full Inside Airbnb listings file.
os.makedirs("figures", exist_ok=True)  # so savefig cells work on a fresh clone
df = pd.read_csv("data/listings.csv")

# --- Adapt the full Inside Airbnb file to the schema this notebook uses ---
# The full file has both a raw `neighbourhood` and the standardized
# `neighbourhood_cleansed`; drop the raw one and use the cleansed values.
df = df.drop(columns=["neighbourhood"])
df = df.rename(columns={"neighbourhood_group_cleansed": "neighbourhood_group",
                        "neighbourhood_cleansed": "neighbourhood"})
# price arrives as text like "$1,200.00"; strip $ and commas -> float.
df["price"] = df["price"].replace(r"[\$,]", "", regex=True).astype(float)

print(df.shape)   # (rows, columns) sanity check
df.head(3)

## 1. Profiling
We look at column types, missing values, and the raw shape of `price` **before** cleaning, so that every cleaning decision below is justified by something we actually observed.

In [ ]:
# dtypes + non-null counts for every column: tells us what is numeric,
# what is text, and where data is missing.
df.info()

# Count missing values, showing only the columns that actually have any.
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

# Summary stats for price. The gap between mean and median, and a min of 0,
# are the two things that drive the cleaning choices in section 2.
print("\nprice summary:")
print(df["price"].describe().round(2))
print("price == 0:", (df["price"] == 0).sum())

## 2. Cleaning

Each decision is driven by what profiling showed:
- **Drop `price == 0` (and blank prices)** — a \$0 nightly rate is a data error, and price is the whole subject of the analysis, so these rows would distort every summary.
- **`reviews_per_month` / `last_review` NaN** — these NaNs line up with listings that have **zero reviews**, so the value is not unknown, it is 0. Fill `reviews_per_month` with 0 and parse `last_review` as a real date.
- **Outliers**: price is extremely right-skewed. Instead of deleting anything, we keep the full cleaned frame `clean` for medians/correlations and build a separate `df_trim` view that drops only the **top 1%** for distribution/relationship plots, so a few very expensive listings don't flatten the axes.
- Leave `name` / `host_name` NaNs alone — they are text labels, not features.


In [ ]:
# Work on a copy so the original df stays available for comparison.
clean = df.copy()

# 1. Drop invalid zero/blank-price rows. A $0/night rate is not real.
clean = clean[clean["price"] > 0]

# 2. Missing reviews_per_month == the listing has no reviews, so fill with 0
#    (not a mean-impute, because 0 is the true value here).
clean["reviews_per_month"] = clean["reviews_per_month"].fillna(0)
#    Parse the review date into a real datetime; unparseable -> NaT.
clean["last_review"] = pd.to_datetime(clean["last_review"], errors="coerce")

# 3. Build a trimmed VIEW for plots only. We compute the 99th percentile of
#    price and keep rows at or below it. This drops the top ~1% long tail so
#    charts are readable, without permanently deleting those rows from `clean`.
p99 = clean["price"].quantile(0.99)
df_trim = clean[clean["price"] <= p99].copy()

# Show how many rows each frame has and how many the trim removed.
print(f"rows: raw={len(df)}, cleaned={len(clean)}, trimmed(<=${p99:.0f})={len(df_trim)}")
print("dropped as top-1% tail:", len(clean) - len(df_trim))

## 3. Price distribution
Raw price is extremely right-skewed. The left panel trims the top 1% so the bulk of listings is visible; the right panel keeps all cleaned data but uses a log scale, which is the standard fix for skew.

In [ ]:
# Two panels side by side: trimmed linear scale, and full data on log scale.
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Left: histogram of trimmed price, with a median line for reference.
ax[0].hist(df_trim["price"], bins=60, color="#3b7dd8", edgecolor="white", linewidth=0.3)
ax[0].axvline(clean["price"].median(), color="#d1495b", ls="--", lw=1.5,
              label=f'median ${clean["price"].median():.0f}')
ax[0].set(title="Price distribution (top 1% trimmed)", xlabel="Price ($/night)", ylabel="Listings")
ax[0].legend()

# Right: log10 of the FULL cleaned price. Taking the log turns the skewed
# distribution into a roughly bell shape, confirming price is ~log-normal.
ax[1].hist(np.log10(clean["price"]), bins=60, color="#3b7dd8", edgecolor="white", linewidth=0.3)
ax[1].set(title="log10(price), full cleaned data", xlabel="log10(price)", ylabel="Listings")

plt.tight_layout()
plt.savefig("figures/01_price_distribution.png", bbox_inches="tight")
plt.show()

## 4. Price by borough (neighbourhood group)
Location is the first suspected driver. We use the **median** because the skew (section 3) makes the mean unreliable for group comparisons.

In [ ]:
# Order boroughs by median price (high to low) so both panels share a ranking.
order = clean.groupby("neighbourhood_group")["price"].median().sort_values(ascending=False).index

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: median price per borough as a labeled bar chart.
med = clean.groupby("neighbourhood_group")["price"].median().reindex(order)
ax[0].bar(med.index, med.values, color="#3b7dd8", edgecolor="white")
for i, v in enumerate(med.values):                       # print the value on each bar
    ax[0].text(i, v + 2, f"${v:.0f}", ha="center", fontsize=9)
ax[0].set(title="Median price by borough", ylabel="Median price ($/night)")

# Right: full spread per borough. showfliers=False hides outliers (already
# summarized elsewhere) so we can compare the boxes themselves. Uses df_trim
# so the y-axis stays in a readable range.
sns.boxplot(data=df_trim, x="neighbourhood_group", y="price", order=order,
            showfliers=False, ax=ax[1], color="#9dc3f0")
ax[1].set(title="Price spread by borough (outliers hidden)", xlabel="", ylabel="Price ($/night)")

plt.tight_layout()
plt.savefig("figures/02_price_by_borough.png", bbox_inches="tight")
plt.show()
med   # display the median table as well

## 5. Price by room type
Same two-panel treatment (median bars + spread) applied to the `room_type` categorical.

In [ ]:
# Rank room types by median price so the panels line up consistently.
rt_order = clean.groupby("room_type")["price"].median().sort_values(ascending=False).index

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: median price per room type, labeled.
med_rt = clean.groupby("room_type")["price"].median().reindex(rt_order)
ax[0].bar(med_rt.index, med_rt.values, color="#e8833a", edgecolor="white")
for i, v in enumerate(med_rt.values):
    ax[0].text(i, v + 2, f"${v:.0f}", ha="center", fontsize=9)
ax[0].set(title="Median price by room type", ylabel="Median price ($/night)")

# Right: spread per room type (outliers hidden, trimmed data for scale).
sns.boxplot(data=df_trim, x="room_type", y="price", order=rt_order,
            showfliers=False, ax=ax[1], color="#f2b784")
ax[1].set(title="Price spread by room type (outliers hidden)", xlabel="", ylabel="Price ($/night)")

plt.tight_layout()
plt.savefig("figures/03_price_by_room_type.png", bbox_inches="tight")
plt.show()
med_rt

## 6. Most and least expensive neighbourhoods
We drill from borough down to individual neighbourhood. A **minimum of 30 listings** per neighbourhood is required so that a block with only a couple of listings can't dominate the ranking on noise.

In [ ]:
# Median price AND listing count per neighbourhood.
g = clean.groupby("neighbourhood")["price"].agg(["median", "count"])
# Keep only neighbourhoods with a decent sample (>=30), then sort by median.
g = g[g["count"] >= 30].sort_values("median")

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True)
bottom = g.head(15)   # 15 cheapest
top = g.tail(15)      # 15 priciest

# Horizontal bars so the long neighbourhood names stay readable.
ax[0].barh(bottom.index, bottom["median"], color="#5aa469")
ax[0].set(title="15 cheapest neighbourhoods (median)", xlabel="Median price ($/night)")
ax[1].barh(top.index, top["median"], color="#d1495b")
ax[1].set(title="15 priciest neighbourhoods (median)", xlabel="Median price ($/night)")

plt.tight_layout()
plt.savefig("figures/04_price_by_neighbourhood.png", bbox_inches="tight")
plt.show()

## 7. Numeric correlations with price
So far we have looked at categorical drivers. Here we check whether any of the **numeric** columns move linearly with price, using a Pearson correlation matrix.

In [ ]:
# The numeric columns worth correlating (ids and coordinates are excluded).
num_cols = ["price", "minimum_nights", "number_of_reviews", "reviews_per_month",
            "calculated_host_listings_count", "availability_365"]
corr = clean[num_cols].corr()   # Pearson correlation matrix

# Heatmap: red = positive, blue = negative, centered at 0. Annotated with the
# actual coefficients so we can read the price row directly.
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlation between numeric features")
plt.tight_layout()
plt.savefig("figures/05_correlation_heatmap.png", bbox_inches="tight")
plt.show()

# Sorted correlations of every feature WITH price. All are near zero, which is
# the key takeaway: no numeric column is a strong linear predictor of price.
corr["price"].sort_values(ascending=False)

## 8. Availability vs price
A raw scatter of tens of thousands of points is unreadable, so we **bin** `availability_365` into ranges and take the median price per bin. This shows the trend without overplotting.


In [ ]:
# Bucket days-available-per-year into interpretable ranges. -1 as the left
# edge lets the first bin capture exactly 0 (fully blocked) listings.
bins = [-1, 0, 90, 180, 270, 365]
labels = ["0 (blocked)", "1-90", "91-180", "181-270", "271-365"]
df_trim["avail_bin"] = pd.cut(df_trim["availability_365"], bins=bins, labels=labels)

# Median price within each availability bucket. observed=True avoids empty
# category warnings from the binned categorical.
med_av = df_trim.groupby("avail_bin", observed=True)["price"].median()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(med_av.index.astype(str), med_av.values, color="#7566b5", edgecolor="white")
for i, v in enumerate(med_av.values):
    ax.text(i, v + 1, f"${v:.0f}", ha="center", fontsize=9)
ax.set(title="Median price by yearly availability", xlabel="Days available per year",
       ylabel="Median price ($/night)")
plt.tight_layout()
plt.savefig("figures/06_availability_vs_price.png", bbox_inches="tight")
plt.show()

## 9. Geographic view
Plotting every listing by longitude/latitude and coloring by price recovers the map of NYC and reveals the spatial price gradient directly, tying together the borough and neighbourhood findings.

In [ ]:
# Scatter each listing at its coordinates; color encodes price.
fig, ax = plt.subplots(figsize=(8, 8))
sc = ax.scatter(df_trim["longitude"], df_trim["latitude"], c=df_trim["price"],
                cmap="viridis", s=4, alpha=0.5,
                vmax=df_trim["price"].quantile(0.95))  # cap color scale at p95 so it isn't washed out
ax.set(title="Listing location colored by price", xlabel="Longitude", ylabel="Latitude")
plt.colorbar(sc, label="Price ($/night)", shrink=0.8)
ax.set_aspect("equal")   # keep the map from stretching
plt.tight_layout()
plt.savefig("figures/07_price_map.png", bbox_inches="tight")
plt.show()

## 10. Observations

1. **Room type is the single strongest lever.** Entire homes/apartments have a median around **\$223/night**, more than double private rooms (about \$103) and well above shared rooms (about \$55). Hotel rooms are highest (about \$447) but are a small slice of listings. Room type separates price more cleanly than any numeric feature.

2. **Location matters a lot, and it stacks on top of room type.** Manhattan is the most expensive borough (median about \$241), followed by Brooklyn (\$151) and Queens (\$120), with Staten Island (\$107) and the Bronx (\$106) the cheapest. At the neighbourhood level the range is much wider. The lat/long map shows a clear price gradient centered on lower/midtown Manhattan.

3. **The numeric features are weak linear predictors of price.** No column in the correlation matrix exceeds about 0.15 in absolute correlation with price. Review counts, minimum nights, and host listing counts barely move price on their own — the strong signals are the categorical ones (room type, location).

4. **Availability has only a mild positive association with price.** Listings available more days per year are slightly more expensive on the median, consistent with professionally managed / entire-home listings, but the effect is small compared to room type and borough.

5. **Price is heavily right-skewed and needs handling.** A small number of listings run into the thousands per night; the median (about \$175) is a far better summary than the mean (about \$278), and log-scaling or trimming the top 1% is necessary for any sensible visualization or modeling.

**Bottom line:** price is best explained by *what you rent* (room type) and *where it is* (borough/neighbourhood). The continuous activity metrics in this dataset add little on their own.
